# سبق 17 - Foundry Local اور Qwen کے ساتھ مقامی AI ایجنٹس بنانا

اس نوٹ بک میں آپ ایک **مقامی انجینئرنگ اسسٹنٹ** بنائیں گے جو مکمل طور پر آپ کے ورک سٹیشن پر چلتا ہے۔ کسی بھی وقت کلاؤڈ انفرنس استعمال نہیں کی جاتی۔ اس اسسٹنٹ کے کام یہ ہیں:

1. **ٹولز کو کال کرنا** Qwen فنکشن کالنگ کے ذریعے Foundry Local کے ذریعے۔
2. **فائلوں کی فہرست بنانا اور پڑھنا** سینڈ باکس شدہ پروجیکٹ ڈائریکٹری کے اندر۔
3. **کوڈ کا تجزیہ کرنا** سادہ میٹرکس کے ساتھ۔
4. **دستاویزات کی تلاش** مقامی RAG (Chroma) کے ساتھ۔
5. **مقامی MCP سرور کا استعمال** (اگر کوئی کنفیگر نہ ہو تو نرم طریقے سے چھوڑ دیا جاتا ہے)۔

ایجنٹ کا کوڈ کلاؤڈ سبقوں کے بالکل مشابہ لگتا ہے — صرف کلائنٹ اینڈ پوائنٹ کلاؤڈ سے `localhost` پر منتقل ہوتا ہے۔


## ترتیب

اس نوٹ بک کو چلانے سے پہلے:

1. **Microsoft Foundry Local انسٹال کریں** (اپنے OS کے لیے [دستاویزات](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) دیکھیں)۔
2. **Qwen ماڈل ڈاؤن لوڈ کریں اور شروع کریں:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. نیچے دیے گئے پائیتھن پیکجز انسٹال کریں۔

سب کچھ مقامی طور پر چلتا ہے۔ تقریباً 8 جی بی ریم والی مشین ایک حقیقت پسندانہ کم از کم ضرورت ہے۔


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. فاؤنڈری لوکل سے کنیکٹ کریں

`FoundryLocalManager` ماڈل کو ضرورت کے مطابق ڈاؤن لوڈ کرتا ہے، لوکل سروس کو شروع کرتا ہے، اور ہمیں **OpenAI-مطابق اینڈپوائنٹ** فراہم کرتا ہے۔ پھر ہم اس کی طرف معیاری OpenAI SDK کو اشارہ کرتے ہیں۔ API کلید ایک لوکل پلیس ہولڈر ہے — اس میں کوئی کلاؤڈ اسناد شامل نہیں ہے۔


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. مقامی ٹولز (سینڈباکسڈ فائل آپریشنز)

ہم ڈسک پر ایک چھوٹا سا نمونہ پروجیکٹ بناتے ہیں، پھر اس پروجیکٹ کی جڑ تک محدود ٹولز کی تعریف کرتے ہیں۔ سینڈباکس چیک یہاں تک کہ مقامی طور پر بھی اہم ہے: ایک ٹول جو بے ترتیب راستے پڑھتا ہے وہ آپ کے صارف کی اجازتوں کے ساتھ چلتا ہے اور آپ جو کچھ بھی کر سکتے ہیں اس کو چھو سکتا ہے۔


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. لوکل RAG ود کروما

ہم دستاویزات کے چند چھوٹے ٹکڑوں کو ایک لوکل کروما کلیکشن میں ایمبیڈ کرتے ہیں۔ کروما اندرونی طور پر چلتا ہے اور ویکٹرز کو ڈسک پر اسٹور کرتا ہے — نہ کوئی سرور، نہ کوئی کلاؤڈ۔ `search_docs` ٹول سوال کے لیے سب سے متعلقہ ٹکڑے حاصل کرتا ہے۔


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. ٹول کال کرنے کا لوپ

اب ہم اوپن اے آئی ٹولز سکیمہ استعمال کرکے ماڈل کے ساتھ ٹولز کو رجسٹر کرتے ہیں اور معیاری ٹول کال کرنے والے لوپ کو چلائیں گے — ماڈل ایک ٹول کی درخواست کرتا ہے، ہم اسے مقامی طور پر چلاتے ہیں، نتیجہ واپس فراہم کرتے ہیں، اور دہرائیں جب تک کہ ماڈل حتمی جواب پیدا نہ کرے۔ قوین کی قابل اعتماد فنکشن کالنگ ہے جو اس کو ڈیوائس پر کام کرنے دیتی ہے۔


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. لوکل MCP (اختیاری)

MCP ایک ٹرانسپورٹ ہے، کلاؤڈ سروس نہیں — ایک MCP سرور مقامی عمل کے طور پر `stdio` پر چل سکتا ہے۔ نیچے دی گئی سیل دکھاتی ہے کہ اگر آپ کے پاس کوئی لوکل MCP سرور ترتیب دیا گیا ہے (مثلاً فائل سسٹم سرور) تو آپ کیسے اس سے جڑیں گے۔ یہ اس وقت gracefully اسکپ کر دیتا ہے جب `LOCAL_MCP_COMMAND` سیٹ نہ ہو، اس لیے نوٹ بک پھر بھی ابتدا سے انتہا تک بغیر اس کے چلتی رہتی ہے۔

سیکورٹی نوٹ: ایک لوکل MCP سرور آپ کے صارف کی اجازتوں کے ساتھ چلتا ہے۔ اسے کسی پروجیکٹ ڈائریکٹری تک محدود کریں اور اس کے نتائج پر عمل کرنے سے پہلے ان کی تصدیق کریں۔


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## خلاصہ

آپ نے ایک انجنئیرنگ اسسٹنٹ تیار کیا ہے جو مکمل طور پر آپ کی مشین پر چلتا ہے:

- **Foundry Local** نے ایک **Qwen** ماڈل کو OpenAI-مطابق اینڈپوائنٹ کے پیچھے فراہم کیا — تاکہ ایجنٹ کوڈ کلاؤڈ سبق کے مطابق ہو۔
- **Sandboxed tools** نے ایجنٹ کو فائل تک رسائی اور کوڈ تجزیہ دیا بغیر پروجیکٹ ڈائریکٹری چھوڑے۔
- **Chroma** نے دستاویزات پر **مقامی RAG** فراہم کیا۔
- **Local MCP** نے مظاہرہ کیا کہ MCP ایکو سسٹم کو آف لائن کیسے دوبارہ استعمال کیا جا سکتا ہے۔

کسی بھی مقام پر کلاؤڈ انفرنس استعمال نہیں ہوئی۔

### چیلنج

مقامی ایجنٹ کو توسیع دیں تاکہ:

1. **متعدد MCP سرورز کے ساتھ کام کرے** — ایک فائل سسٹم سرور اور گٹ سرور کو جوڑے اور ایجنٹ کو ان میں سے انتخاب کرنے دیں۔
2. **مقامی میموری کا استعمال کرے** — ایک مختصر گفتگو کی تاریخ ڈسک پر محفوظ کرے تاکہ اسسٹنٹ نوٹ بک ری اسٹارٹس کے دوران پچھلے مراحل کو یاد رکھ سکے۔
3. **مقامی کثیر-ایجنٹ تنظیم کی حمایت کرے** — ایک دوسرا مقامی ایجنٹ (مثلاً ایک جائزہ لینے والا) شامل کریں اور دونوں کو مل کر کسی کام پر تعاون کرنے دیں۔

اگلے سبق میں آپ سیکھیں گے کہ تعینات AI ایجنٹس کو کیسے محفوظ کیا جائے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
